# Imports

In [1]:
import os
import time
import json
import pandas as pd
import ollama
from nlp_spo_plus.data.cvrp_text import genLatentData, build_node_prompt

# Configuration

In [2]:
num_data = 2500
num_clients = 10
num_vehicules = 3
poly_degree = 4
noise_width = 0.1
seed = 135

In [3]:
output_dir = os.path.join("..", "..", "data", "synthetic", "textual")
os.makedirs(output_dir, exist_ok=True)
output_csv = os.path.join(output_dir, "dataset_cvrp_nlp.csv")

# Data Generation

In [4]:
print("generating latent states and ground truth costs...")
latent_features, costs, demands, capacity = genLatentData(
    num_data=num_data,
    num_clients=num_clients,
    num_vehicules=num_vehicules,
    deg=poly_degree,
    noise_width=noise_width,
    seed=seed
)

generating latent states and ground truth costs...


# Text Generation

In [ ]:
dataset_records = []
start_idx = 0

if os.path.exists(output_csv):
    try:
        existing_df = pd.read_csv(output_csv)
        dataset_records = existing_df.to_dict('records')
        start_idx = len(dataset_records)
        print(f"Found existing dataset. Resuming generation from instance {start_idx}...")
    except Exception as e:
        print("Could not read existing CSV, starting from scratch.")

# text generation via local ollama
model_name = "qwen2.5:7b"
print(f"starting local text generation using {model_name}...")

for instance_idx in range(start_idx, num_data):
    instance_nodes_states = latent_features[instance_idx]
    instance_node_texts = []
    
    for node_idx, node_state in enumerate(instance_nodes_states):
        prompt = build_node_prompt(node_state)
        
        # Robust Retry Mechanism
        max_retries = 3
        for attempt in range(max_retries):
            try:
                response = ollama.chat(model=model_name, messages=[
                    {
                        "role": "system", 
                        "content": "You are a professional local traffic reporter. Write short, fluent, natural text respecting all instructions without meta-commentary."
                    },
                    {
                        "role": "user", 
                        "content": prompt
                    }
                ], options={
                    "num_ctx": 512 # Optimise the VRAM, we only need small context
                })
                node_text = response['message']['content'].strip().replace("\n", " ")
                break
                
            except Exception as e:
                print(f"\n[Warning] API Error on instance {instance_idx}, node {node_idx} (Attempt {attempt+1}/{max_retries}): {e}")
                if attempt < max_retries - 1:
                    print("Cooling down for 5 seconds before retrying...")
                    time.sleep(5)
                else:
                    print("CRITICAL FAILURE after 3 attempts.")
                    raise e 
            
        instance_node_texts.append(node_text)
        
    dataset_records.append({
        "instance_id": instance_idx,
        "node_texts": json.dumps(instance_node_texts),
        "costs": json.dumps(costs[instance_idx].tolist()),
        "demands": json.dumps(demands.tolist()),
        "capacity": capacity
    })
    
    # Checkpointing : on sauvegarde le CSV toutes les 50 instances
    if (instance_idx + 1) % 50 == 0:
        pd.DataFrame(dataset_records).to_csv(output_csv, index=False)
        print(f"Checkpoint: instance {instance_idx + 1}/{num_data} successfully processed and saved to disk.")

starting local text generation using qwen2.5:7b...
Checkpoint: instance 50/2500 successfully processed and saved to disk.
Checkpoint: instance 100/2500 successfully processed and saved to disk.
Checkpoint: instance 150/2500 successfully processed and saved to disk.
Checkpoint: instance 200/2500 successfully processed and saved to disk.
Checkpoint: instance 250/2500 successfully processed and saved to disk.
Checkpoint: instance 300/2500 successfully processed and saved to disk.
Checkpoint: instance 350/2500 successfully processed and saved to disk.
Checkpoint: instance 400/2500 successfully processed and saved to disk.
Checkpoint: instance 450/2500 successfully processed and saved to disk.
Checkpoint: instance 500/2500 successfully processed and saved to disk.
Checkpoint: instance 550/2500 successfully processed and saved to disk.
Checkpoint: instance 600/2500 successfully processed and saved to disk.
Checkpoint: instance 650/2500 successfully processed and saved to disk.
Checkpoint: in

In [6]:
df = pd.DataFrame(dataset_records)
df.to_csv(output_csv, index=False)
print(f"\ndataset successfully generated and saved to {output_csv}")


dataset successfully generated and saved to ..\..\data\synthetic\textual\dataset_cvrp_nlp.csv
